In [ ]:
import scanpy as sc
import pandas as pd
from pathlib import Path
import numpy as np

In [ ]:
# -------------------------
# Configuration
# -------------------------
# Change this variable to target any other "unpredictable" protein
target_protein = "CD45RA" 

input_h5ad = "../data/adt.h5ad"
response_dir = Path("../data/response")
output_dir = Path("../data")



In [ ]:
# -------------------------
# Processing
# -------------------------
response_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

# 1) Load ADT
adata = sc.read_h5ad(input_h5ad)
print(f"Loaded {input_h5ad} with {adata.n_obs} cells and {adata.n_vars} proteins.")

# 2) Check target exists
if target_protein not in adata.var_names:
  raise ValueError(
      f"Protein '{target_protein}' not found. "
      f"Example available: {adata.var_names.tolist()[:10]}..."
  )

# 3) Extract response from **norm layer** (correct)
if "norm" not in adata.layers:
  raise KeyError("Expected adt.layers['norm'] to exist for the regression target.")

j = adata.var_names.get_loc(target_protein)
col = adata.layers["norm"][:, j]
y_response = col.toarray().ravel() if hasattr(col, "toarray") else np.asarray(col).ravel()
y_response = y_response.astype(np.float32, copy=False)

# Create clean filename
clean_name = target_protein.replace(".", "_").replace("-", "_")
response_path = response_dir / f"{clean_name}.csv"

# Save response with cell IDs as index
pd.DataFrame({target_protein: y_response}, index=adata.obs_names.astype(str)).to_csv(response_path)
print(f"Separated {target_protein} (from layers['norm']) into {response_path}")

# 4) Remove target from feature set (inputs) to prevent leakage
remaining_features = adata.var_names[adata.var_names != target_protein]
adata_filtered = adata[:, remaining_features].copy()

# 5) Save the modified h5ad (keeps layers, including 'norm' for remaining proteins)
output_filename = output_dir / f"adt_minus_{clean_name}.h5ad"
adata_filtered.write_h5ad(output_filename)

print(f"Saved remaining {adata_filtered.n_vars} proteins to {output_filename}")